# Data Preprocessing

### 1. Data Loading

In [33]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

In [34]:
from pathlib import Path

pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd().resolve().parents[0]
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TAXONOMY_PATH = DATA_DIR / "label_taxonomy.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission_random_guess.csv"

In [35]:
# Verify files exist
required_paths = [TRAIN_PATH, TEST_PATH, TAXONOMY_PATH]
missing_files = [str(p) for p in required_paths if not p.exists()]

if missing_files:
    print("Missing required files:")
    for file in missing_files:
        print("-", file)
    raise FileNotFoundError("Please add the dataset CSV files to the data folder.")

# Load data with tab separator
train_df = pd.read_csv(TRAIN_PATH, sep="\t")
test_df = pd.read_csv(TEST_PATH, sep="\t")
taxonomy_df = pd.read_csv(TAXONOMY_PATH, sep="\t")

# Load sample submission if it exists
if SAMPLE_SUBMISSION_PATH.exists():
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
else:
    sample_submission = None

print("\n✓ Data loaded successfully!")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Taxonomy shape: {taxonomy_df.shape}")
print(f"Sample submission shape: {sample_submission.shape if sample_submission is not None else 'N/A'}")

# Display first few rows
print("\nFirst few rows of train data:")
display(train_df.head())

print("\nFirst few rows of test data:")
display(test_df.head())

print("\nLabel taxonomy:")
display(taxonomy_df.head())

print("\n Sample submission:")
display(sample_submission.head())


✓ Data loaded successfully!
Train shape: (139156, 3)
Test shape: (34790, 2)
Taxonomy shape: (39, 2)
Sample submission shape: (34790, 2)

First few rows of train data:


,id,abstract,label_id
0,95829,"Project-based learning plays a crucial role in computing education. However, its open-ended nature makes tracking project development and assessing success ...",14
1,73195,Edge computing is a distributed computing paradigm that brings computation\nand data storage closer to the user's geographical location to improve respons...,10
2,22319,"In today's computing environment, where Artificial Intelligence (AI) and data\nprocessing are moving toward the Internet of Things (IoT) and Edge computin...",3
3,76837,"Diffusion pipelines, renowned for their powerful visual generation capabilities, have seen widespread adoption in generative vision tasks (e.g., text-to-ima...",10
4,159150,Convolutional neural networks (CNNs) are emerging as powerful tools for image\nprocessing in important commercial applications. We focus on the important\...,30



First few rows of test data:


,id,abstract
0,173148,"Non-Boolean computing based on emerging post-CMOS technologies can\npotentially pave the way for low-power neural computing platforms. However,\nexisting ..."
1,29098,"Combining the properties of monovariate internal functions as proposed in\nKolmogorov superimposition theorem, in tandem with the bounds wielded by the\nm..."
2,28211,"We study the problem of unsupervised discovery and segmentation of object\nparts, which, as an intermediate local representation, are capable of finding\n..."
3,136101,We revisit the fundamental problem of I/O-efficiently computing $r$-way\nseparators on planar graphs. An $r$-way separator divides a planar graph with\n$N...
4,97133,"Through a mixed-method analysis of data from Scratch, we examine how novices\nlearn to program with simple data structures by using community-produced\nle..."



Label taxonomy:


,label_id,category
0,0,cs.DS
1,1,cs.CC
2,2,cs.CR
3,3,cs.PF
4,4,cs.CV



 Sample submission:


,id,label_id
0,173148,38
1,29098,28
2,28211,14
3,136101,7
4,97133,20


In [36]:
# Check for missing values
print(f"\nMissing values in train: {train_df.isnull().sum().sum()}")
print(f"Missing values in test: {test_df.isnull().sum().sum()}")


Missing values in train: 0
Missing values in test: 0


### 2. Text Cleaning

In [37]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

class TextPreprocessor:
    def __init__(self, use_stemming=False, use_lemmatization=True, remove_stopwords=True):
        self.use_stemming = use_stemming
        self.use_lemmatization = use_lemmatization
        self.remove_stopwords = remove_stopwords
        
        if use_stemming:
            self.stemmer = PorterStemmer()
        if use_lemmatization:
            self.lemmatizer = WordNetLemmatizer()
        if remove_stopwords:
            self.stop_words = set(stopwords.words('english'))
    
    def clean_text(self, text):
        """Basic text cleaning"""
        # Convert to lowercase
        text = text.lower()
        
        # Remove special characters and digits (keep only letters and spaces)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        
        # Remove extra whitespace
        text = ' '.join(text.split())
        
        return text
    
    def preprocess(self, text):
        """Full preprocessing pipeline"""
        # Clean text
        text = self.clean_text(text)
        
        # Tokenize
        words = text.split()
        
        # Remove stopwords
        if self.remove_stopwords:
            words = [word for word in words if word not in self.stop_words]
        
        # Apply stemming or lemmatization
        if self.use_stemming:
            words = [self.stemmer.stem(word) for word in words]
        elif self.use_lemmatization:
            words = [self.lemmatizer.lemmatize(word) for word in words]
        
        # Join back into text
        return ' '.join(words)

# Apply preprocessing to abstracts
preprocessor = TextPreprocessor(use_lemmatization=True, remove_stopwords=True)

train_df['clean_abstract'] = train_df['abstract'].apply(preprocessor.preprocess)
test_df['clean_abstract'] = test_df['abstract'].apply(preprocessor.preprocess)

print("Sample before preprocessing:")
print(train_df['abstract'].iloc[0][:200])
print("\nSample after preprocessing:")
print(train_df['clean_abstract'].iloc[0][:200])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\esthe\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\esthe\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\esthe\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


Sample before preprocessing:
Project-based learning plays a crucial role in computing education. However, its open-ended nature makes tracking project development and assessing success challenging. We investigate how dialogue and

Sample after preprocessing:
projectbased learning play crucial role computing education however openended nature make tracking project development assessing success challenging investigate dialogue system interaction log predict


### 3. Prepare Data for All Tasks

In [39]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    train_df['cleaned_abstract'],
    train_df['label_id'],
    test_size=0.2,
    random_state=42,
    stratify=train_df['label_id']  # Important for imbalanced data
)

print(f"Training samples: {len(X_train_raw)}")
print(f"Validation samples: {len(X_val_raw)}")

# Label Encoding
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

print(f"\nNumber of classes: {len(label_encoder.classes_)}")
print(f"Classes: {label_encoder.classes_[:10]}...")  # Show first 10 classes

Training samples: 111324
Validation samples: 27832

Number of classes: 39
Classes: [0 1 2 3 4 5 6 7 8 9]...


### 4. TF-IDF Vectorization

In [40]:
tfidf_task1 = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),      # Unigrams and bigrams
    min_df=2,                # Ignore terms appearing in < 2 docs
    max_df=0.8,              # Ignore terms appearing in > 80% of docs
    sublinear_tf=True,       # Use 1+log(tf)
    stop_words='english'     # Remove common English stop words
)

# Fit on training data only
print("Fitting TF-IDF on training data...")
X_train_tfidf = tfidf_task1.fit_transform(X_train_raw)
X_val_tfidf = tfidf_task1.transform(X_val_raw)
X_test_tfidf = tfidf_task1.transform(test_df['cleaned_abstract'])

print(f"Train TF-IDF shape: {X_train_tfidf.shape}")
print(f"Validation TF-IDF shape: {X_val_tfidf.shape}")
print(f"Test TF-IDF shape: {X_test_tfidf.shape}")

# Convert to dense arrays for logistic regression (from scratch)
# Note: This might be memory intensive, but 5000 features * 5000 samples is manageable
X_train_dense_task1 = X_train_tfidf.toarray()
X_val_dense_task1 = X_val_tfidf.toarray()
X_test_dense_task1 = X_test_tfidf.toarray()

print(f"\nDense arrays created - Train: {X_train_dense_task1.shape}, Test: {X_test_dense_task1.shape}")

Fitting TF-IDF on training data...
Train TF-IDF shape: (111324, 5000)
Validation TF-IDF shape: (27832, 5000)
Test TF-IDF shape: (34790, 5000)

Dense arrays created - Train: (111324, 5000), Test: (34790, 5000)


### 5. Label Encoding

In [41]:
# Encode labels for training
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

print(f"Encoded classes: {label_encoder.classes_}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Encoded classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38]
Number of classes: 39


In [43]:
class ArxivPreprocessor:
    """Complete preprocessing pipeline for ArXiv data"""
    
    def __init__(self, max_features=5000, test_size=0.2):
        self.max_features = max_features
        self.test_size = test_size
        self.text_preprocessor = TextPreprocessor()
        self.tfidf = TfidfVectorizer(max_features=max_features)
        self.label_encoder = LabelEncoder()
        self.scaler = StandardScaler(with_mean=False)
        
    def fit(self, train_df):
        """Fit preprocessor on training data"""
        # Preprocess text
        train_df['clean_abstract'] = train_df['abstract'].apply(
            self.text_preprocessor.preprocess
        )
        
        # Fit TF-IDF
        X = self.tfidf.fit_transform(train_df['clean_abstract'])
        
        # Fit label encoder
        y = self.label_encoder.fit_transform(train_df['label_id'])
        
        # Fit scaler
        X_dense = X.toarray()
        self.scaler.fit(X_dense)
        
        return X_dense, y
    
    def transform(self, df):
        """Transform new data"""
        # Preprocess text
        df['clean_abstract'] = df['abstract'].apply(
            self.text_preprocessor.preprocess
        )
        
        # Transform TF-IDF
        X = self.tfidf.transform(df['clean_abstract'])
        
        # Scale
        X_dense = X.toarray()
        X_scaled = self.scaler.transform(X_dense)
        
        return X_scaled
    
    def fit_transform(self, train_df):
        """Fit and transform training data"""
        self.fit(train_df)
        return self.transform(train_df)

# Usage
preprocessor = ArxivPreprocessor(max_features=5000)
X_train_processed, y_train_encoded = preprocessor.fit_transform(train_df)
X_test_processed = preprocessor.transform(test_df)

ValueError: too many values to unpack (expected 2)